# Face Search on Video - Standalone Processing Notebook

This notebook contains all the necessary code to process videos for face and bib search on Google Colab.

## Instructions
1. Mount Google Drive.
2. Prepare a folder in your Drive with your videos (e.g., `face-search/data/videos`).
3. Update the `BASE_DIR` variable in the configuration cell to point to your `face-search` folder.
4. Run all cells.

## 1. Mount Drive & Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies
!pip install fastapi uvicorn opencv-python insightface onnxruntime faiss-cpu numpy python-multipart paddlepaddle paddleocr sentence-transformers
!apt-get update && apt-get install -y libgl1-mesa-glx

## 2. Configuration & Imports

In [ ]:
import os
import cv2
import numpy as np
import json
import faiss
from typing import List, Dict, Optional, Tuple
from glob import glob
from tqdm import tqdm
import insightface
from paddleocr import PaddleOCR
from sentence_transformers import SentenceTransformer

# --- CONFIGURATION ---
# Update this path to where your project/data folder is in Google Drive
BASE_DIR = "/content/drive/MyDrive/face-search-on-video-mvp"

class Settings:
    # InsightFace
    DET_SIZE = (640, 640)
    MODEL_ROOT = os.path.join(BASE_DIR, "data/insightface_models")
    
    # FAISS
    VECTOR_DIM = 512
    INDEX_PATH = os.path.join(BASE_DIR, "data/index/index.faiss")
    METADATA_PATH = os.path.join(BASE_DIR, "data/index/metadata.json")
    
    # OCR & Text Search
    BIB_INDEX_PATH = os.path.join(BASE_DIR, "data/index/bib_index.faiss")
    BIB_METADATA_PATH = os.path.join(BASE_DIR, "data/index/bib_metadata.json")
    TEXT_EMBEDDING_MODEL = "all-MiniLM-L6-v2"
    TEXT_EMBEDDING_DIM = 384
    
    # Video Processing
    FRAME_INTERVAL = 30
    VIDEO_DIR = os.path.join(BASE_DIR, "data/videos")

settings = Settings()

# Ensure directories exist
os.makedirs(os.path.dirname(settings.INDEX_PATH), exist_ok=True)
os.makedirs(settings.MODEL_ROOT, exist_ok=True)
os.makedirs(settings.VIDEO_DIR, exist_ok=True)

## 3. Core Classes

In [ ]:
class FaceProcessor:
    def __init__(self):
        self.app = insightface.app.FaceAnalysis(name='buffalo_l', root=settings.MODEL_ROOT)
        self.app.prepare(ctx_id=0, det_size=settings.DET_SIZE)

    def get_embedding(self, img_array: np.ndarray) -> List[dict]:
        faces = self.app.get(img_array)
        results = []
        for face in faces:
             results.append({
                 'embedding': face.embedding,
                 'bbox': face.bbox.astype(int).tolist(),
                 'det_score': float(face.det_score)
             })
        return results

class OCRProcessor:
    def __init__(self, lang: str = 'en'):
        self.ocr = PaddleOCR(use_angle_cls=True, lang=lang, show_log=False, use_gpu=True)

    def get_text(self, img_array: np.ndarray) -> List[Dict]:
        result = self.ocr.ocr(img_array, cls=True)
        output = []
        if result and result[0]:
            for line in result[0]:
                points = line[0]
                text, score = line[1]
                output.append({
                    'text': text,
                    'bbox': points,
                    'score': score
                })
        return output

class VectorStore:
    def __init__(self, load_existing: bool = True):
        self.dimension = settings.VECTOR_DIM
        self.index = None
        self.metadata = []
        
        if load_existing and os.path.exists(settings.INDEX_PATH):
            self.load()
        else:
            self.index = faiss.IndexFlatIP(self.dimension)
            self.metadata = []
            self.bib_index = faiss.IndexFlatIP(settings.TEXT_EMBEDDING_DIM)
            self.bib_metadata = []

    def add_vectors(self, vectors: np.ndarray, metadata_entries: List[Dict]):
        if vectors.shape[0] != len(metadata_entries):
            raise ValueError("Number of vectors and metadata entries must match.")
        faiss.normalize_L2(vectors)
        self.index.add(vectors)
        self.metadata.extend(metadata_entries)

    def add_bib_vectors(self, vectors: np.ndarray, metadata_entries: List[Dict]):
        if vectors.shape[0] != len(metadata_entries):
            raise ValueError("Number of vectors and metadata entries must match.")
        faiss.normalize_L2(vectors)
        self.bib_index.add(vectors)
        self.bib_metadata.extend(metadata_entries)

    def save(self):
        os.makedirs(os.path.dirname(settings.INDEX_PATH), exist_ok=True)
        faiss.write_index(self.index, settings.INDEX_PATH)
        with open(settings.METADATA_PATH, 'w') as f:
            json.dump(self.metadata, f)
        
        if hasattr(self, 'bib_index') and self.bib_index:
             faiss.write_index(self.bib_index, settings.BIB_INDEX_PATH)
        if hasattr(self, 'bib_metadata'):
             with open(settings.BIB_METADATA_PATH, 'w') as f:
                json.dump(self.bib_metadata, f)

    def load(self):
        print(f"Loading index from {settings.INDEX_PATH}")
        self.index = faiss.read_index(settings.INDEX_PATH)
        if os.path.exists(settings.METADATA_PATH):
            with open(settings.METADATA_PATH, 'r') as f:
                self.metadata = json.load(f)
        else:
            self.metadata = []

        if os.path.exists(settings.BIB_INDEX_PATH):
            print(f"Loading bib index from {settings.BIB_INDEX_PATH}")
            self.bib_index = faiss.read_index(settings.BIB_INDEX_PATH)
            if os.path.exists(settings.BIB_METADATA_PATH):
                with open(settings.BIB_METADATA_PATH, 'r') as f:
                    self.bib_metadata = json.load(f)
            else:
                 self.bib_metadata = []
        else:
            self.bib_index = faiss.IndexFlatIP(settings.TEXT_EMBEDDING_DIM)
            self.bib_metadata = []

## 4. Processing Logic

In [ ]:
def process_video(video_path: str, face_processor: FaceProcessor, ocr_processor: OCRProcessor, text_model: SentenceTransformer, vector_store: VectorStore):
    print(f"Processing {video_path}...")
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error opening video file {video_path}")
        return

    frame_count = 0
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    process_interval = settings.FRAME_INTERVAL 
    
    pbar = tqdm(total=total_frames)
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        frame_count += 1
        pbar.update(1)
        
        if frame_count % process_interval != 0:
            continue
            
        timestamp = frame_count / fps

        # --- Face Detection ---
        try:
            faces = face_processor.get_embedding(frame)
            if faces:
                embeddings = []
                metadata_list = []
                for face in faces:
                    embeddings.append(face['embedding'])
                    metadata_list.append({
                        'video_path': video_path.replace(BASE_DIR, "data"), # Store relative path for portability or adjust as needed
                        'timestamp': timestamp,
                        'frame_idx': frame_count,
                        'bbox': face['bbox'],
                        'det_score': face['det_score'],
                        'type': 'face'
                    })
                if embeddings:
                    vector_store.add_vectors(np.array(embeddings), metadata_list)
        except Exception as e:
            print(f"Error processing face frame {frame_count}: {e}")

        # --- OCR / Bib Detection ---
        try:
            texts = ocr_processor.get_text(frame)
            if texts:
                bib_embeddings = []
                bib_metadata_list = []
                
                for item in texts:
                    text_str = item['text']
                    if any(c.isdigit() for c in text_str):
                        emb = text_model.encode([text_str])[0]
                        bib_embeddings.append(emb)
                        bib_metadata_list.append({
                            'video_path': video_path.replace(BASE_DIR, "data"),
                            'timestamp': timestamp,
                            'frame_idx': frame_count,
                            'bbox': item['bbox'],
                            'score': item['score'],
                            'text': text_str,
                            'type': 'bib'
                        })
                
                if bib_embeddings:
                     vector_store.add_bib_vectors(np.array(bib_embeddings), bib_metadata_list)
        except Exception as e:
            print(f"Error processing OCR frame {frame_count}: {e}")
            
    cap.release()
    pbar.close()

## 5. Main Execution
Run this cell to start processing.

In [ ]:
def main():
    # --- BATCH CONFIGURATION ---
    BATCH_SIZE = 5 # Process 5 videos per shard
    
    print("Configuration:")
    print(f"  BASE_DIR: {BASE_DIR}")
    print(f"  VIDEO_DIR: {settings.VIDEO_DIR}")
    print(f"  BATCH_SIZE: {BATCH_SIZE}")

    print("Initializing models...")
    face_processor = FaceProcessor()
    ocr_processor = OCRProcessor()
    text_model = SentenceTransformer(settings.TEXT_EMBEDDING_MODEL)
    
    video_files = glob(os.path.join(settings.VIDEO_DIR, "*.mp4")) + \
                  glob(os.path.join(settings.VIDEO_DIR, "*.avi")) + \
                  glob(os.path.join(settings.VIDEO_DIR, "*.mov"))
    
    if not video_files:
        print(f"No videos found in {settings.VIDEO_DIR}")
        return
    
    print(f"Found {len(video_files)} videos.")
    
    # Split videos into batches
    for i in range(0, len(video_files), BATCH_SIZE):
        batch_idx = i // BATCH_SIZE
        batch_videos = video_files[i : i + BATCH_SIZE]
        
        print(f"\n--- Processing Batch {batch_idx} ({len(batch_videos)} videos) ---")
        
        # Initialize VectorStore for this batch (shard)
        shard_dir = os.path.join(BASE_DIR, f"data/index/shard_{batch_idx}")
        os.makedirs(shard_dir, exist_ok=True)
        
        settings.INDEX_PATH = os.path.join(shard_dir, "index.faiss")
        settings.METADATA_PATH = os.path.join(shard_dir, "metadata.json")
        settings.BIB_INDEX_PATH = os.path.join(shard_dir, "bib_index.faiss")
        settings.BIB_METADATA_PATH = os.path.join(shard_dir, "bib_metadata.json")
        
        vector_store = VectorStore(load_existing=False) 
        
        for video_file in batch_videos:
            process_video(video_file, face_processor, ocr_processor, text_model, vector_store)
            
        vector_store.save()
        print(f"Batch {batch_idx} saved to {shard_dir}")

    print("All batches processed.")

main()